# NTP Temporal Calibration

# In this notebook ……
We characterize clock synchronization between the agents of one recorded run (a rosbag2 MCAP):

1. extract the light topics of the bag (NTP status, events, and a header-stamp audit of every topic),
2. find out who is the NTP server and who are the clients,
3. compute the clock offset statistics of every client and detect clock steps,
4. cross-check the offsets with an NTP-independent measurement (header stamp vs. recorder time),
5. produce the figure and the filled-in *Temporal Calibration* paragraph for the paper.

In [ ]:
%pip install mcap mcap-ros2-support numpy pandas pyarrow matplotlib tabulate

In [ ]:
import os
import sys
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.insert(0, os.path.abspath('.'))          # so extract_bag.py next to this notebook is importable
from extract_bag import extract

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

In [ ]:
def node_of(topic):
    """'/mobile_1/ntp/status' -> 'mobile_1'."""
    return topic.strip('/').split('/')[0]


def ecdf(x):
    """
    Input:
        x: 1-D array-like
    Output:
        xs: sorted values (NaN removed)
        ys: empirical CDF at each value
    """
    xs = np.sort(np.asarray(x, dtype=float))
    xs = xs[~np.isnan(xs)]
    ys = np.arange(1, len(xs) + 1) / max(len(xs), 1)
    return xs, ys


def load_topic(extracts, pattern):
    """
    Input:
        extracts: directory written by extract_bag.py
        pattern: glob pattern of the parquet files to load, e.g. '*ntp__status.parquet'
    Output:
        df: concatenated DataFrame with a 'topic' column, or None if nothing matched
    """
    frames = []
    for f in sorted(glob.glob(os.path.join(extracts, pattern))):
        df = pd.read_parquet(f)
        df['topic'] = '/' + os.path.splitext(os.path.basename(f))[0].replace('__', '/')
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else None

# 1. Parameters

## 1.1 Paths

In [ ]:
RUN      = 'coop2'
BAG      = './mirc_dataset_coop2_20260828_completed_0.mcap'   # the rosbag2 MCAP file
EXTRACTS = os.path.join('./extracts', RUN)                    # per-topic parquet tables go here
OUT      = os.path.join('./results', RUN, 'ntp')              # tables, figure and LaTeX go here
os.makedirs(OUT, exist_ok=True)

## 1.2 Thresholds

In [ ]:
STEP_THRESHOLD_MS = 1.0    # |offset_delta| above this counts as a clock step
SENSOR_PERIOD_MS  = None   # None = use the shortest median period found in the stamp audit

## 1.3 Colors
One fixed color per agent, reused in every figure of the dataset paper.

In [ ]:
AGENT_COLOR = {'mobile_1': '#2a78d6', 'mobile_2': '#eb6834', 'infra_1': '#1baf7a'}
STEP_COLOR  = '#e34948'
EVENT_COLOR = '#eda100'
TEXT, TEXT2, GRID = '#0b0b0b', '#52514e', '#e6e5e1'

def color_for(node):
    return AGENT_COLOR.get(node, '#4a3aa7')

# 2. Extract Bag

## 2.1 Run the extraction
Decodes every topic except images, point clouds, laser scans, camera info, paths and raw Ouster packets, and writes one parquet table per topic plus `stamp_audit.parquet` (header stamp, publish time and log time of **every** message of every header-bearing topic, parsed from the raw bytes). Skipped when the extracts already exist; set `FORCE_EXTRACT = True` to redo it.

In [ ]:
FORCE_EXTRACT = False

if FORCE_EXTRACT or not os.path.exists(os.path.join(EXTRACTS, 'metadata.json')):
    meta = extract(BAG, EXTRACTS)
else:
    meta = json.load(open(os.path.join(EXTRACTS, 'metadata.json')))
    print('using existing extracts in', EXTRACTS)

## 2.2 Topics found in the bag

In [ ]:
topics = pd.DataFrame([{'topic': t, 'type': v['type'], 'count': v['count'],
                        'extracted': t in meta['extracted']} for t, v in meta['topics'].items()])
topics.sort_values('topic').reset_index(drop=True)

# 3. Load NTP Data

## 3.1 NTP status

In [ ]:
ntp = load_topic(EXTRACTS, '*ntp__status.parquet')
assert ntp is not None, 'no *ntp__status.parquet under ' + EXTRACTS

ntp['node']            = ntp['topic'].map(node_of)
ntp['offset_ms']       = ntp['offset_seconds'] * 1e3
ntp['delay_ms']        = ntp['delay_seconds'] * 1e3
ntp['jitter_ms']       = ntp['jitter_seconds'] * 1e3
ntp['offset_delta_ms'] = ntp['offset_delta_seconds'] * 1e3
ntp['clock_stepped']   = ntp['clock_stepped'].astype(bool)
print(len(ntp), 'rows from', ntp['topic'].nunique(), 'topics')
ntp.head(3).T

## 3.2 NTP events

In [ ]:
events = load_topic(EXTRACTS, '*ntp__events.parquet')
print(0 if events is None else len(events), 'event rows')

## 3.3 Stamp audit
Header stamp / publish time / recorder log time of every message of every header-bearing topic.

In [ ]:
audit_path = os.path.join(EXTRACTS, 'stamp_audit.parquet')
audit = pd.read_parquet(audit_path).dropna(subset=['header_stamp_ns']) if os.path.exists(audit_path) else None
print(0 if audit is None else len(audit), 'audit rows')

## 3.4 Common time axis
Seconds since the first recorded message of the run.

In [ ]:
t0_ns = int(ntp['log_time_ns'].min())
if audit is not None and len(audit):
    t0_ns = min(t0_ns, int(audit['log_time_ns'].min()))

ntp['t_s'] = (ntp['log_time_ns'] - t0_ns) / 1e9
if events is not None:
    events['t_s'] = (events['log_time_ns'] - t0_ns) / 1e9
if audit is not None:
    audit['t_s'] = (audit['log_time_ns'] - t0_ns) / 1e9
print('run duration seen by NTP topics: %.1f s' % ntp['t_s'].max())

# 4. NTP Roles
`sync_source` names the server each client follows. `mobile_1` publishes no NTP topic in this bag, so it should appear here as the sync source of the others.

In [ ]:
def get_roles(ntp):
    """
    Input:
        ntp: NTP status DataFrame (section 3)
    Output:
        roles: one row per (topic, role, hostname, sync_source) with count, stratum, rate and time span
    """
    roles = (ntp.groupby(['topic', 'role', 'hostname', 'sync_source'], dropna=False)
                .agg(n=('seq', 'size'),
                     stratum=('stratum', lambda s: int(s.mode().iloc[0])),
                     synchronized_frac=('synchronized', 'mean'),
                     rate_hz=('t_s', lambda t: (len(t) - 1) / max(t.max() - t.min(), 1e-9)),
                     t_first_s=('t_s', 'min'),
                     t_last_s=('t_s', 'max'))
                .reset_index())
    return roles

In [ ]:
roles = get_roles(ntp)
roles.to_csv(os.path.join(OUT, 'ntp_roles.csv'), index=False)
roles

# 5. Clock Offset

## 5.1 Offset statistics per client

In [ ]:
def offset_statistics(ntp, step_threshold_ms, run):
    """
    Input:
        ntp: NTP status DataFrame (section 3)
        step_threshold_ms: |offset_delta| above this counts as a clock step
        run: run name written into the table
    Output:
        summary: one row per (topic, role, hostname) with offset / delay / jitter statistics and clock steps
        series: dict label -> per-client DataFrame sorted by time (used for plotting)
    """
    rows, series = [], {}
    for (topic, role, host), g in ntp.groupby(['topic', 'role', 'hostname']):
        g = g.sort_values('t_s')
        label = host if role == 'client' else '%s (%s)' % (host, role)
        series[label] = g
        abs_off = g['offset_ms'].abs()
        steps_flag  = int((g['clock_stepped'] & ~g['clock_stepped'].shift(1, fill_value=False)).sum())
        steps_delta = int((g['offset_delta_ms'].abs() > step_threshold_ms).sum())
        warn = sorted({w for ws in g['warnings'] for w in (list(ws) if ws is not None else [])})
        rows.append(dict(
            run=run, topic=topic, node=node_of(topic), role=role, hostname=host,
            sync_source=g['sync_source'].mode().iloc[0], stratum=int(g['stratum'].mode().iloc[0]), n=len(g),
            duration_s=g['t_s'].max() - g['t_s'].min(),
            offset_mean_ms=g['offset_ms'].mean(), offset_median_ms=g['offset_ms'].median(), offset_std_ms=g['offset_ms'].std(),
            abs_offset_mean_ms=abs_off.mean(), abs_offset_p95_ms=abs_off.quantile(0.95), abs_offset_max_ms=abs_off.max(),
            t_of_max_abs_offset_s=g.loc[abs_off.idxmax(), 't_s'],
            delay_median_ms=g['delay_ms'].median(), delay_max_ms=g['delay_ms'].max(),
            jitter_median_ms=g['jitter_ms'].median(),
            root_dispersion_median_ms=g['root_dispersion'].median() * 1e3,
            freq_error_mean_ppm=g['frequency_error_ppm'].mean(),
            poll_interval_mode_s=int(g['poll_interval_seconds'].mode().iloc[0]),
            reach_min=int(g['reach_register'].min()), reachability_min_pct=int(g['reachability_percent'].min()),
            synchronized_frac=g['synchronized'].astype(bool).mean(),
            clock_steps_flagged=steps_flag, clock_steps_by_delta=steps_delta,
            step_times_s=json.dumps([round(float(x), 2) for x in g.loc[g['clock_stepped'], 't_s']][:20]),
            warnings='; '.join(warn)))
    return pd.DataFrame(rows), series

In [ ]:
summary, series = offset_statistics(ntp, STEP_THRESHOLD_MS, RUN)
summary.to_csv(os.path.join(OUT, 'ntp_summary.csv'), index=False)

SHOW = ['hostname', 'role', 'sync_source', 'stratum', 'n', 'offset_mean_ms', 'offset_median_ms',
        'abs_offset_p95_ms', 'abs_offset_max_ms', 'delay_median_ms', 'jitter_median_ms',
        'poll_interval_mode_s', 'reach_min', 'clock_steps_flagged', 'clock_steps_by_delta']
summary[SHOW].round(3)

## 5.2 Clock steps and NTP events

In [ ]:
for label, g in series.items():
    steps = g.loc[g['clock_stepped'], 't_s'].round(2).tolist()
    big   = g.loc[g['offset_delta_ms'].abs() > STEP_THRESHOLD_MS, 't_s'].round(2).tolist()
    print('%-22s flagged steps at t = %s   |offset_delta| > %.1f ms at t = %s' % (label, steps, STEP_THRESHOLD_MS, big))

if events is not None and len(events):
    display(events[['t_s', 'topic', 'data']])
else:
    print('no NTP event messages in this run')

## 5.3 Offset over time

In [ ]:
plt.figure(figsize=(12, 4))
for label, g in series.items():
    plt.plot(g['t_s'], g['offset_ms'], lw=1.2, color=color_for(g['node'].iloc[0]), label=label)
    for ts in g.loc[g['clock_stepped'], 't_s']:
        plt.axvline(ts, color=STEP_COLOR, lw=0.8, alpha=0.8)
if events is not None and len(events):
    for t in events['t_s']:
        plt.axvspan(t - 0.4, t + 0.4, color=EVENT_COLOR, alpha=0.35, lw=0)
plt.axhline(0, color=GRID, lw=0.8)
plt.xlabel('time in run [s]')
plt.ylabel('NTP offset to server [ms]')
plt.title('client clock offset (red = clock step, yellow = NTP event)')
plt.legend(frameon=False)
plt.grid(True, color=GRID, lw=0.5)

## 5.4 Delay and jitter over time

In [ ]:
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
for label, g in series.items():
    plt.plot(g['t_s'], g['delay_ms'], lw=1, color=color_for(g['node'].iloc[0]), label=label)
plt.xlabel('time in run [s]'); plt.ylabel('round-trip delay [ms]'); plt.title('delay'); plt.legend(frameon=False); plt.grid(True, color=GRID, lw=0.5)
plt.subplot(1, 2, 2)
for label, g in series.items():
    plt.plot(g['t_s'], g['jitter_ms'], lw=1, color=color_for(g['node'].iloc[0]), label=label)
plt.xlabel('time in run [s]'); plt.ylabel('jitter [ms]'); plt.title('jitter'); plt.grid(True, color=GRID, lw=0.5)

# 6. Offset Distribution

In [ ]:
plt.figure(figsize=(6, 4))
for label, g in series.items():
    x, y = ecdf(g['offset_ms'].abs())
    plt.plot(x, y, lw=1.5, color=color_for(g['node'].iloc[0]), label=label)
plt.xscale('log')
plt.xlabel('|offset| [ms]'); plt.ylabel('ECDF'); plt.title('|offset| distribution per client')
plt.legend(frameon=False); plt.grid(True, color=GRID, lw=0.5, which='both')

# 7. NTP-Independent Check
Every sensor is stamped in software on arrival at its host, so `header.stamp − log_time` is transport latency plus the clock offset between that node and the recorder. In the recorder's own clock it is slightly negative (pure latency). A node whose topics sit systematically off the recorder's has a clock offset of that size, independent of what the NTP monitor reports.

## 7.1 Per-topic statistics

In [ ]:
def audit_statistics(audit):
    """
    Input:
        audit: stamp audit DataFrame (section 3.3)
    Output:
        audit_df: one row per (node, topic) with the median message period and the
                  median / p05 / p95 / IQR of header stamp minus recorder log time
    """
    rows = []
    for (node, topic), g in audit.groupby(['node', 'topic']):
        g = g.sort_values('header_stamp_ns')
        dt = np.diff(g['header_stamp_ns'].to_numpy()) / 1e6
        d = g['stamp_minus_log_ms']
        rows.append(dict(node=node, topic=topic, type=g['type'].iloc[0], n=len(g),
                         period_median_ms=float(np.median(dt)) if len(dt) > 10 else np.nan,
                         stamp_minus_log_median_ms=d.median(),
                         stamp_minus_log_p05_ms=d.quantile(0.05),
                         stamp_minus_log_p95_ms=d.quantile(0.95),
                         stamp_minus_log_iqr_ms=d.quantile(0.75) - d.quantile(0.25)))
    return pd.DataFrame(rows).sort_values(['node', 'topic']).reset_index(drop=True)

In [ ]:
audit_df = audit_statistics(audit) if audit is not None and len(audit) else None
if audit_df is not None:
    audit_df.to_csv(os.path.join(OUT, 'ntp_audit.csv'), index=False)
    display(audit_df.round(3))

## 7.2 Per-node offset relative to the recorder

In [ ]:
if audit_df is not None:
    node_med = audit_df.groupby('node')['stamp_minus_log_median_ms'].median().round(3).to_frame('median_of_topic_medians_ms')
    display(node_med)

## 7.3 Shortest sensor period
The sync bound: every client offset should stay below this.

In [ ]:
sensor_period_ms, fastest = SENSOR_PERIOD_MS, '(user supplied)'
if SENSOR_PERIOD_MS is None and audit_df is not None:
    valid = audit_df[(audit_df['n'] >= 100) & audit_df['period_median_ms'].notna()]
    if len(valid):
        sensor_period_ms = float(valid['period_median_ms'].min())
        fastest = valid.loc[valid['period_median_ms'].idxmin(), 'topic']
print('shortest sensor period: %s ms (%s)' % (sensor_period_ms, fastest))
for r in summary.itertuples():
    print('%-10s max |offset| %.2f ms = %.2f x shortest period' % (r.hostname, r.abs_offset_max_ms, r.abs_offset_max_ms / sensor_period_ms))

## 7.4 Header stamp − log time per topic

In [ ]:
plt.figure(figsize=(6, 4))
if audit is not None and len(audit):
    seen = set()
    for (node, topic), g in audit.groupby(['node', 'topic']):
        if len(g) < 50:
            continue
        x, y = ecdf(g['stamp_minus_log_ms'])
        plt.plot(x, y, lw=0.9, color=color_for(node), label=node if node not in seen else None)
        seen.add(node)
plt.xlabel('header stamp − log time [ms]'); plt.ylabel('ECDF (one line per topic)')
plt.title('NTP-independent check'); plt.legend(frameon=False); plt.grid(True, color=GRID, lw=0.5)

# 8. Figure for the Paper
(a) client clock offset over time, (b) ECDF of |offset| with the shortest sensor period, (c) header stamp − log time per topic.

In [ ]:
plt.rcParams.update({'font.size': 8, 'axes.edgecolor': GRID, 'axes.labelcolor': TEXT,
                     'xtick.color': TEXT2, 'ytick.color': TEXT2, 'text.color': TEXT})
fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.4), constrained_layout=True)

ax = axes[0]
for label, g in series.items():
    ax.plot(g['t_s'], g['offset_ms'], lw=1.2, color=color_for(g['node'].iloc[0]), label=label)
    for ts in g.loc[g['clock_stepped'], 't_s']:
        ax.axvline(ts, color=STEP_COLOR, lw=0.8, alpha=0.8)
if events is not None and len(events):
    for t in events['t_s']:
        ax.axvspan(t - 0.4, t + 0.4, color=EVENT_COLOR, alpha=0.35, lw=0)
ax.axhline(0, color=GRID, lw=0.8)
ax.set_xlabel('time in run [s]'); ax.set_ylabel('NTP offset to server [ms]')
ax.set_title('(a) client clock offset', loc='left', fontsize=8); ax.legend(frameon=False, fontsize=7); ax.grid(True, color=GRID, lw=0.5)

ax = axes[1]
for label, g in series.items():
    x, y = ecdf(g['offset_ms'].abs())
    ax.plot(x, y, lw=1.5, color=color_for(g['node'].iloc[0]), label=label)
if sensor_period_ms:
    ax.axvline(sensor_period_ms, color=TEXT2, lw=0.8, ls='--')
    ax.text(sensor_period_ms, 0.05, ' shortest sensor\n period %.1f ms' % sensor_period_ms, fontsize=6.5, color=TEXT2, va='bottom')
ax.set_xscale('log'); ax.set_xlabel('|offset| [ms]'); ax.set_ylabel('ECDF')
ax.set_title('(b) offset distribution', loc='left', fontsize=8); ax.grid(True, color=GRID, lw=0.5, which='both')

ax = axes[2]
if audit is not None and len(audit):
    seen = set()
    for (node, topic), g in audit.groupby(['node', 'topic']):
        if len(g) < 50:
            continue
        x, y = ecdf(g['stamp_minus_log_ms'])
        ax.plot(x, y, lw=0.9, color=color_for(node), label=node if node not in seen else None)
        seen.add(node)
    ax.set_xlabel('header stamp − log time [ms]'); ax.set_ylabel('ECDF (one line per topic)'); ax.legend(frameon=False, fontsize=7)
ax.set_title('(c) NTP-independent check', loc='left', fontsize=8); ax.grid(True, color=GRID, lw=0.5)

fig.savefig(os.path.join(OUT, 'fig_ntp.pdf'))
fig.savefig(os.path.join(OUT, 'fig_ntp.png'), dpi=200)
plt.rcParams.update(plt.rcParamsDefault)

# 9. Paper Text
The *Temporal Calibration* subsection with the numbers of this run filled in.

In [ ]:
def ntp_subsection(run, roles, summary, audit_df, sensor_period_ms):
    """
    Input:
        run: run name
        roles, summary, audit_df: tables from sections 4, 5 and 7
        sensor_period_ms: the sync bound from section 7.3
    Output:
        tex: LaTeX text of the NTP Synchronization subsubsection
    """
    tt = lambda s: '\\texttt{' + str(s).replace('_', '\\_') + '}'
    clients = summary[summary['role'].str.contains('client')]
    server  = clients['sync_source'].mode().iloc[0] if len(clients) else '?'
    strata  = ', '.join(str(s) for s in sorted(clients['stratum'].unique()))
    rate    = ', '.join('%.1f' % r for r in roles['rate_hz'])
    parts = ['%s had a mean offset of %.2f\\,ms (mean $|\\cdot|$ %.2f\\,ms, 95th percentile %.2f\\,ms, maximum %.2f\\,ms) '
             'with a median round-trip delay of %.2f\\,ms and %d clock step%s'
             % (tt(r.hostname), r.offset_mean_ms, r.abs_offset_mean_ms, r.abs_offset_p95_ms, r.abs_offset_max_ms,
                r.delay_median_ms, r.clock_steps_flagged, '' if r.clock_steps_flagged == 1 else 's')
             for r in clients.itertuples()]
    max_all = clients['abs_offset_max_ms'].max()
    audit_sentence = bound_sentence = ''
    if audit_df is not None:
        spread = audit_df.groupby('node')['stamp_minus_log_median_ms'].median()
        audit_sentence = (' As an NTP-independent check, the per-node median of header stamp minus recorder receive time '
                          'spans %.2f to %.2f\\,ms across nodes, bounding clock offset plus transport latency.'
                          % (spread.min(), spread.max()))
    if sensor_period_ms:
        if max_all < sensor_period_ms:
            bound_sentence = (' All offsets are below the shortest sensor period in the recording (%.1f\\,ms), so cross-agent '
                              'messages can be associated by timestamp without further alignment.' % sensor_period_ms)
        else:
            bound_sentence = (' The maximum offset exceeds the shortest sensor period (%.1f\\,ms); cross-agent association '
                              'of the fastest topics needs the recorded offsets applied.' % sensor_period_ms)
    tex = ('\\subsubsection{NTP Synchronization}\n'
           'All agents are synchronized over NTP on the shared wireless network.\n'
           '%s acts as the NTP server and the other agents synchronize to it as\n'
           'stratum-%s clients; each client publishes its NTP state at about %s\\,Hz throughout every run.\n'
           'Over run %s, %s.%s%s\n'
           'No sensor is hardware-triggered or hardware-timestamped: every message is stamped in software by\n'
           'its driver on arrival at the host, so the header stamps carry the NTP-aligned host clock plus the\n'
           "driver's arrival latency, and the offsets above bound clock disagreement between agents, not\n"
           'sensor exposure time.\n'
           % (tt(server), strata, rate, tt(run), '; '.join(parts), audit_sentence, bound_sentence))
    return tex

In [ ]:
tex = ntp_subsection(RUN, roles, summary, audit_df, sensor_period_ms)
open(os.path.join(OUT, 'ntp_subsection.tex'), 'w').write(tex)
print(tex)

In [ ]:
print('outputs in', OUT)
print(sorted(os.listdir(OUT)))